In [1]:
from simulation import citygraph_dataset
from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

### Стартовый набор маршрутов

In [2]:
dataset_name = 'mandl'
demand_time_weight = 0.33
route_time_weight = 0.33
median_connectivity_weight = 0.33
experiment_name = f'Kemerovo_exp_{dataset_name}_pp_{demand_time_weight}_op_{route_time_weight}_cp_{median_connectivity_weight}'
initial_routes_name = experiment_name+ '_starting'
generated_routes_name = experiment_name + '_generated'

In [3]:
common_dict = {
    'experiment': {
        'anomaly': False,
        'cpu': False,
        'seed': 0,
        'symmetric_routes': True,
        'cost_function': {
            'type': 'mine',
            'kwargs': {
                'mean_stop_time_s': 0,
                'avg_transfer_wait_time_s': 300,
                'demand_time_weight': demand_time_weight,
                'route_time_weight': route_time_weight,
                'median_connectivity_weight': median_connectivity_weight,
                'constraint_violation_weight': 5.0,
                'variable_weights': True,
                'pp_fraction': 0.15,
                'op_fraction': 0.15,
                'mcw_fraction': 0.15,
            }
        }
    },

    'model': {
        'common': {
            'dropout': 0.0,
            'nonlin_type': 'ReLU',
            'embed_dim': 64,
        },
        'route_generator': {
            'kwargs': {
                'force_linking_unlinked': False,
                'logit_clip': None,
                'n_nodepair_layers': 3,
                'n_pathscorer_layers': 3,
                'pathscorer_hidden_dim': 16,
                'n_halt_layers': 3,
                'halt_scorer_type': 'endpoints',
                'serial_halting': True,
            },
            'type': 'PathCombiningRouteGenerator',
        },
        'backbone_gn': {
            'net_type': 'graph attn',
            'kwargs': {
                'n_layers': 5,
                'in_node_dim': 4,
                'in_edge_dim': 14,
                'use_norm': False,
                'n_heads': 4,
                'dense': False,
            }
        },
        'weights': '../TNDP_learning/output/inductive_random_graphs.pt',
    },

    'eval': {
        'csv': True,
        'n_routes': 6,
        'min_route_len': 2,
        'max_route_len': 8,
        'dataset': {
            'type': 'tensor',
        }
    },
}


In [4]:
cfg_eval_dict = dict(common_dict)  # поверхностная копия

# копируем вложенные словари, чтобы не портить common_dict
cfg_eval_dict['experiment'] = dict(common_dict['experiment'])
cfg_eval_dict['experiment']['cost_function'] = dict(common_dict['experiment']['cost_function'])
cfg_eval_dict['experiment']['cost_function']['kwargs'] = dict(common_dict['experiment']['cost_function']['kwargs'])

# переопределения
cfg_eval_dict['experiment']['logdir'] = None
cfg_eval_dict['experiment']['cost_function']['kwargs']['use_weighted_connectivity'] = True

cfg_eval_dict['n_samples'] = 50
cfg_eval_dict['batch_size'] = 16
cfg_eval_dict['run_name'] = initial_routes_name


In [5]:
cfg_neural_dict = dict(common_dict)  # поверхностная копия

# копируем вложенные словари
cfg_neural_dict['experiment'] = dict(common_dict['experiment'])
cfg_neural_dict['experiment']['cost_function'] = dict(common_dict['experiment']['cost_function'])
cfg_neural_dict['experiment']['cost_function']['kwargs'] = dict(common_dict['experiment']['cost_function']['kwargs'])

# переопределения
cfg_neural_dict['experiment']['logdir'] = 'training_logs'

cfg_neural_dict['n_bees'] = 10
cfg_neural_dict['n_iterations'] = 100
cfg_neural_dict['batch_size'] = 10
cfg_neural_dict['neural_bees'] = True
cfg_neural_dict['force_linking_unlinked'] = False

cfg_neural_dict['init'] = {
    'method': 'load',
    'path': f'output_routes/nn_construction_{initial_routes_name}_routes.pkl',
}

cfg_neural_dict['run_name'] = generated_routes_name


In [6]:
cfg_eval = OmegaConf.create(cfg_eval_dict)
cfg_neural = OmegaConf.create(cfg_neural_dict)


In [7]:
import torch
import numpy as np
from pathlib import Path

# Пути к файлам
instances_dir = "CEC2013Supp/Instances"
instance_name = "Mandl"

data_dir = Path(instances_dir)

# Чтение координат
coords_path = data_dir / (instance_name + 'Coords.txt')
node_locs = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)
orig_pos = torch.tensor(np.genfromtxt(coords_path, skip_header=1), dtype=torch.float32)

# Чтение времени путешествия
tt_path = data_dir / (instance_name + 'TravelTimes.txt')
street_adj = torch.tensor(np.genfromtxt(tt_path), dtype=torch.float32) * 60

# Чтение спроса
dmd_path = data_dir / (instance_name + 'Demand.txt')
demand = torch.tensor(np.genfromtxt(dmd_path), dtype=torch.float32)

# Результаты
print("Coords tensor shape:", node_locs.shape)
print("Travel times tensor shape:", street_adj.shape)
print("Demand tensor shape:", demand.shape)

Coords tensor shape: torch.Size([15, 2])
Travel times tensor shape: torch.Size([15, 15])
Demand tensor shape: torch.Size([15, 15])


In [8]:
input_tensors = {            
    'street_adj':street_adj,
    'demand':demand,    
    'node_locs':node_locs
    }

In [9]:
metrics, unserved_demand = main_eval(cfg_eval, tensors=input_tensors)

/root/TNDP_learning/learning/utils.py:321: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand,  metrics)


In [10]:
unserved_demand

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      

### Генерация 

In [11]:
metrics, unserved_demand = main_bee(cfg_neural, input_tensors)

In [20]:
unserved_demand

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      

In [21]:
metrics

{'cost': tensor([0.5036]),
 'ATT': tensor([13.4310]),
 'RTT': tensor([142.]),
 '$d_0$': tensor([59.2164]),
 '$d_1$': tensor([29.2229]),
 '$d_2$': tensor([11.2396]),
 '$d_{un}$': tensor([0.3211]),
 '# disconnected node pairs': tensor([0.]),
 '# stops out of bounds': tensor([0.]),
 'median_connectivity': tensor([13.2667]),
 'median_connectivity_weighted': tensor([94.9894]),
 'average wall-clock duration': tensor([54.1750]),
 'average # iterations': tensor([101.])}

In [14]:
metrics['median_connectivity'] /= 60 

In [15]:
# Порядок нужных ключей
keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']

print('\t'.join(str(round(metrics[key].item(),3)) for key in keys_order))
print('\t'.join(str(round(metrics[key].item(),3)) for key in keys_order))

13.431	142.0	13.267	0.504	0.321	59.216	29.223	11.24
13.431	142.0	13.267	0.504	0.321	59.216	29.223	11.24


In [16]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from learning.eval_route_generator import main as main_eval
from learning.bee_colony import main as main_bee
from tqdm import tqdm

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

# Все параметры экспериментов
experiments = [
    ("mandl", 1, 0, 0),
    ("mandl", 0, 1, 0),
    ("mandl", 0, 0, 1),
    ("mandl", 0.5, 0.5, 0),
    ("mandl", 0.5, 0, 0.5),
    ("mandl", 0, 0.5, 0.5),
    ("mandl", 0.33, 0.33, 0.33),
    ("mumford0", 1, 0, 0),
    ("mumford0", 0, 1, 0),
    ("mumford0", 0, 0, 1),
    ("mumford0", 0.5, 0.5, 0),
    ("mumford0", 0.5, 0, 0.5),
    ("mumford0", 0, 0.5, 0.5),
    ("mumford0", 0.33, 0.33, 0.33),
    ("mumford1", 1, 0, 0),
    ("mumford1", 0, 1, 0),
    ("mumford1", 0, 0, 1),
    ("mumford1", 0.5, 0.5, 0),
    ("mumford1", 0.5, 0, 0.5),
    ("mumford1", 0, 0.5, 0.5),
    ("mumford1", 0.33, 0.33, 0.33),
    ("mumford2", 1, 0, 0),
    ("mumford2", 0, 1, 0),
    ("mumford2", 0, 0, 1),
    ("mumford2", 0.5, 0.5, 0),
    ("mumford2", 0.5, 0, 0.5),
    ("mumford2", 0, 0.5, 0.5),
    ("mumford2", 0.33, 0.33, 0.33),
    ("mumford3", 1, 0, 0),
    ("mumford3", 0, 1, 0),
    ("mumford3", 0, 0, 1),
    ("mumford3", 0.5, 0.5, 0),
    ("mumford3", 0.5, 0, 0.5),
    ("mumford3", 0, 0.5, 0.5),
    ("mumford3", 0.33, 0.33, 0.33),
]

# Путь к весам модели
model_weights_path = "../TNDP_learning/output/inductive_random_graphs_checkpoints/iter990.pt"

# CSV файл для результатов
results_file = Path("experiment_results.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов с tqdm
for dataset_name, dt, rt, ct in tqdm(experiments, desc="Running experiments"):
    try:
        experiment_name = f"exp_{dataset_name}_pp_{dt}_op_{rt}_cp_{ct}"
        initial_routes_name = experiment_name + '_starting'
        generated_routes_name = experiment_name + '_generated'

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={initial_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}"
                ]
            )
        eval_metrics, unserved_demand = main_eval(cfg_eval)

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_bee = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={generated_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        bee_metrics, unserved_demand = main_bee(cfg_bee)
        bee_metrics['median_connectivity'] /= 60

        keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name.lower(), dt, rt, ct] + [round(bee_metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

    except Exception as e:
        print(f"[✗] Failed {experiment_name}: {e}")

Running experiments:   0%|          | 0/35 [00:00<?, ?it/s]

Running experiments:   6%|▌         | 2/35 [00:00<00:02, 13.95it/s]

[✗] Failed exp_mandl_pp_1_op_0_cp_0: main() missing 1 required positional argument: 'tensors'
[✗] Failed exp_mandl_pp_0_op_1_cp_0: main() missing 1 required positional argument: 'tensors'


Running experiments:  11%|█▏        | 4/35 [00:00<00:01, 15.59it/s]

[✗] Failed exp_mandl_pp_0_op_0_cp_1: main() missing 1 required positional argument: 'tensors'
[✗] Failed exp_mandl_pp_0.5_op_0.5_cp_0: main() missing 1 required positional argument: 'tensors'


...

[✗] Failed exp_mumford3_pp_0_op_0_cp_1: main() missing 1 required positional argument: 'tensors'
[✗] Failed exp_mumford3_pp_0.5_op_0.5_cp_0: main() missing 1 required positional argument: 'tensors'


Running experiments:  97%|█████████▋| 34/35 [00:02<00:00, 17.01it/s]

[✗] Failed exp_mumford3_pp_0.5_op_0_cp_0.5: main() missing 1 required positional argument: 'tensors'
[✗] Failed exp_mumford3_pp_0_op_0.5_cp_0.5: main() missing 1 required positional argument: 'tensors'


Running experiments: 100%|██████████| 35/35 [00:02<00:00, 16.61it/s]

[✗] Failed exp_mumford3_pp_0.33_op_0.33_cp_0.33: main() missing 1 required positional argument: 'tensors'


In [17]:
# Эксперименты только с mandl
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm

model_weights_path = "../TNDP_learning/output/inductive_random_graphs_checkpoints/iter990.pt"

mandl_experiments = [
    ("mandl", 1, 0, 0),
    ("mandl", 0, 1, 0),
    ("mandl", 0, 0, 1),
    ("mandl", 0.5, 0.5, 0),
    ("mandl", 0.5, 0, 0.5),
    ("mandl", 0, 0.5, 0.5),
    ("mandl", 0.33, 0.33, 0.33),
]

# CSV файл
results_file = Path("eval_only_mandl.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов только main_eval
for dataset_name, dt, rt, ct in tqdm(mandl_experiments, desc="Evaluating mandl"):
    try:
        run_name = f"mandl_eval_pp_{dt}_op_{rt}_cp_{ct}"

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={run_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                ]
            )

        metrics = main_eval(cfg_eval)
        keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name, dt, rt, ct] + [round(metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

    except Exception as e:
        print(f"[✗] Failed eval for {run_name}: {e}")

Evaluating mandl:   0%|          | 0/7 [00:00<?, ?it/s]

[✗] Failed eval for mandl_eval_pp_1_op_0_cp_0: main() missing 1 required positional argument: 'tensors'


Evaluating mandl:  29%|██▊       | 2/7 [00:00<00:00, 16.73it/s]

[✗] Failed eval for mandl_eval_pp_0_op_1_cp_0: main() missing 1 required positional argument: 'tensors'
[✗] Failed eval for mandl_eval_pp_0_op_0_cp_1: main() missing 1 required positional argument: 'tensors'


Evaluating mandl:  57%|█████▋    | 4/7 [00:00<00:00, 17.00it/s]

[✗] Failed eval for mandl_eval_pp_0.5_op_0.5_cp_0: main() missing 1 required positional argument: 'tensors'
[✗] Failed eval for mandl_eval_pp_0.5_op_0_cp_0.5: main() missing 1 required positional argument: 'tensors'


Evaluating mandl: 100%|██████████| 7/7 [00:00<00:00, 16.80it/s]

[✗] Failed eval for mandl_eval_pp_0_op_0.5_cp_0.5: main() missing 1 required positional argument: 'tensors'
[✗] Failed eval for mandl_eval_pp_0.33_op_0.33_cp_0.33: main() missing 1 required positional argument: 'tensors'
